# Computed metrics

In [16]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from scipy.stats import pearsonr
from statannotations.Annotator import Annotator
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

metrics_df = pd.read_csv("../data/good/master_data_12k.csv", usecols=["team", "global_id", "sequence"])

In [17]:
df = pd.read_csv("../data/good/cpu_boltz_rosetta_stats.csv", usecols=["name", "B_hs_engaged", "C_hs_engaged", "all_hs_engaged", "iplddt", "pae", "pde"])
df["global_id"] = df["name"].apply(lambda x: int(x.split("_")[0].split("binder")[-1]))
df = df.drop(columns=["name"])
df.columns = ["boltz_B_hs_engaged", "boltz_C_hs_engaged", "boltz_all_hs_engaged", "boltz_hs_iplddt", "boltz_hs_pae", "boltz_hs_pde", "global_id"]

# folds that completely failed or went to wrong side got NaN, fill with 0
df["boltz_hs_iplddt"] = df["boltz_hs_iplddt"].fillna(0)
df["boltz_hs_pae"] = df["boltz_hs_pae"].fillna(0)
df["boltz_hs_pde"] = df["boltz_hs_pde"].fillna(0)

metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [18]:
df = pd.read_csv("../data/good/ipsae_stats.csv")
df["global_id"] = df["model"].apply(lambda x: int(x.split("_")[0].split("binder")[-1]))
df = df.drop(columns=["model"])
df = df.rename(columns={col: f"boltz_{col}" for col in df.columns if col != "global_id"})
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [19]:
df = pd.read_csv("../data/good/thread_boltz_rosetta_stats.csv")
df["global_id"] = df["name"].apply(lambda x: int(x.split("_")[0].split("binder")[-1]))
# df = df.drop(columns=["name", "boltz", "scores", "pdb", "name.1", "description"])
df = df.drop(columns=["name"])
# prepend "boltz_rosetta_" in front of cols except global_id
df.columns = [f"boltz_rosetta_{col}" if col != "global_id" else col for col in df.columns]

# set NaN to 1e9 for these boltz rosetta scores
cols_to_fill = [
    'boltz_rosetta_A_BC_complex_normalized',
    'boltz_rosetta_A_BC_side1_normalized',
    'boltz_rosetta_A_BC_side2_normalized',
    'boltz_rosetta_A_B_complex_normalized',
    'boltz_rosetta_A_B_side1_normalized',
    'boltz_rosetta_A_B_side2_normalized',
    'boltz_rosetta_A_C_complex_normalized',
    'boltz_rosetta_A_C_side1_normalized',
    'boltz_rosetta_A_C_side2_normalized',
    'boltz_rosetta_complex_normalized',
    'boltz_rosetta_side1_normalized',
    'boltz_rosetta_side2_normalized'
]
for col in cols_to_fill:
    df[col] = df[col].fillna(1e9)


metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [20]:
df = pd.read_csv("../data/good/axis_angles_aterm_vs_bc.csv", usecols=["file","angle_Cterm_deg","angle_Nterm_deg"])
df["global_id"] = df["file"].apply(lambda x: x.split("/")[-1].split("_")[0].split("binder")[-1]).astype(int)
df = df.sort_values(by="global_id")
df = df.drop(columns=["file"])
# prepend "boltz_" in front of cols except global_id
df.columns = [f"boltz_{col}" if col != "global_id" else col for col in df.columns]
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [21]:
df = pd.read_csv("../data/good/mmseqs/all_vs_cd20ab.tsv", sep="\t", names=["query","target","pident","alnlen","qlen","tlen","qcov","tcov","evalue"])
df["global_id"] = df["query"].apply(lambda x: int(x.split("|")[0]))
df = df.drop(columns=["query", "alnlen", "qlen", "tlen", "qcov", "tcov", "evalue"])
df.columns = ["closest_ab", "closest_ab_pident", "global_id"]
metrics_df = metrics_df.merge(df, on="global_id", how="left")
metrics_df["closest_ab_pident"] = metrics_df["closest_ab_pident"].fillna(0)
metrics_df["closest_ab"] = metrics_df["closest_ab"].fillna("none")

In [22]:
df = pd.read_csv("../data/good/mmseqs/all_vs_ubiquitin.tsv", sep="\t", names=["query","target","pident","alnlen","qlen","tlen","qcov","tcov","evalue"])
df["global_id"] = df["query"].apply(lambda x: int(x.split("|")[0]))
df = df.drop(columns=["query", "target", "alnlen", "qlen", "tlen", "qcov", "tcov", "evalue"])
df.columns = ["ubiquitin_pident", "global_id"]
metrics_df = metrics_df.merge(df, on="global_id", how="left")
metrics_df["ubiquitin_pident"] = metrics_df["ubiquitin_pident"].fillna(0)

In [23]:
df = pd.read_csv("../data/good/max_pairwise_identities_hamming.csv")
df = df.drop(columns=["team", "sequence"])
# append _hamming to all columns except global_id
df = df.rename(columns={col: f"{col}_hamming" for col in df.columns if col != "global_id"})
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [24]:
df = pd.read_csv("../data/good/max_pairwise_identities_mmseqs.csv")
df = df.drop(columns=["seq_group"])
# append _hamming to all columns except global_id
df = df.rename(columns={col: f"{col}_mmseqs" for col in df.columns if col != "global_id"})
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [25]:
df = pd.read_csv("../data/jakub/12k_all_jakub.csv")
for col in df.columns:
    if "esm2" in col:
        # normalize
        df[col] = df[col] / df["sequence_x"].str.len()
df = df.drop(columns=["sequence_x", "sequence_y"])
df = df.rename(columns={col: col.replace("chai_", "chai1_") for col in df.columns})
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [26]:
df = pd.read_csv("../data/jakub/disulfide_and_pipi_contacts.csv")
df = df.drop(columns=["pdb_path", "disulfide_within_chain", "disulfide_between_chain", "pipi_within_chain", "pipi_between_chain"])
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [27]:
df = pd.read_csv("../data/good/binder_sapscore.csv")
df["global_id"] = df["basename"].apply(lambda x: int(x.split("binder")[-1].split("_")[0]))
df = df.drop(columns=["basename"])
df.columns = ["boltz_sap_score", "global_id"]
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [28]:
df = pd.read_csv("../data/good/consolidated_mpnn_scores_pivoted.csv")
# prepend "boltz_" to all columns except global_id
df.columns = [f"boltz_{col}" if col != "global_id" else col for col in df.columns]

# fill na with 0 for all boltz mpnn scores. These are only for the 32 seqs with X in them

metrics_df = metrics_df.merge(df, on="global_id", how="left")
metrics_df["boltz_autoregwithoutseq_proteinmpnn"] = metrics_df["boltz_autoregwithoutseq_proteinmpnn"].fillna(0)
metrics_df["boltz_autoregwithoutseq_solublempnn"] = metrics_df["boltz_autoregwithoutseq_solublempnn"].fillna(0)
metrics_df["boltz_autoregwithseq_proteinmpnn"] = metrics_df["boltz_autoregwithseq_proteinmpnn"].fillna(0)
metrics_df["boltz_autoregwithseq_solublempnn"] = metrics_df["boltz_autoregwithseq_solublempnn"].fillna(0)
metrics_df["boltz_singleaascorewithoutseq_proteinmpnn"] = metrics_df["boltz_singleaascorewithoutseq_proteinmpnn"].fillna(0)
metrics_df["boltz_singleaascorewithoutseq_solublempnn"] = metrics_df["boltz_singleaascorewithoutseq_solublempnn"].fillna(0)
metrics_df["boltz_singleaascorewithseq_proteinmpnn"] = metrics_df["boltz_singleaascorewithseq_proteinmpnn"].fillna(0)
metrics_df["boltz_singleaascorewithseq_solublempnn"] = metrics_df["boltz_singleaascorewithseq_solublempnn"].fillna(0)

In [29]:
df = pd.read_csv("../data/good/leah_12k_dssp.csv")
metrics_df = metrics_df.merge(df, on="global_id", how="left")

In [30]:
# # Helical wheel moments (max per binder)
# from pathlib import Path

# helix_df = pd.read_csv("../data/good/helical_moments.csv")
# helix_df["global_id"] = helix_df["file"].apply(
#     lambda x: int(Path(x).name.split("_")[0].replace("binder", ""))
# )

# # Overall stats (all helices)
# helix_agg_all = (
#     helix_df.groupby("global_id").agg({
#         "hydrophobic_moment": ["max", "mean", "min"],
#         "charge_moment": ["max", "mean", "min"],
#         "net_charge_total": ["max", "mean", "min"]
#     })
#     .reset_index()
# )
# helix_agg_all.columns = ["global_id", "boltz_max_hydrophobic_moment", "boltz_mean_hydrophobic_moment", "boltz_min_hydrophobic_moment", "boltz_max_charge_moment", "boltz_mean_charge_moment", "boltz_min_charge_moment", "boltz_max_net_charge_total", "boltz_mean_net_charge_total", "boltz_min_net_charge_total"]

# # Helix length statistics
# helix_len_stats = (
#     helix_df.groupby("global_id")["helix_length"]
#     .agg(["max", "mean", "sum"])
#     .reset_index()
#     .rename(columns={"max": "boltz_max_helix_length", "mean": "boltz_mean_helix_length", "sum": "boltz_total_helix_length"})
# )

# # Long helices only (>= 20 residues)
# helix_long = helix_df[helix_df["helix_length"] >= 20]
# helix_agg_long = (
#     helix_long.groupby("global_id").agg({
#         "hydrophobic_moment": ["max", "mean", "min"],
#         "charge_moment": ["max", "mean", "min"],
#         "net_charge_total": ["max", "mean", "min"]
#     })
#     .reset_index()
# )
# helix_agg_long.columns = ["global_id", "boltz_max_hydrophobic_moment_long", "boltz_mean_hydrophobic_moment_long", "boltz_min_hydrophobic_moment_long", "boltz_max_charge_moment_long", "boltz_mean_charge_moment_long", "boltz_min_charge_moment_long", "boltz_max_net_charge_total_long", "boltz_mean_net_charge_total_long", "boltz_min_net_charge_total_long"]

# # Helix count
# helix_counts = helix_df.groupby("global_id").size().reset_index(name="boltz_helical_wheel_count")
# helix_counts_long = helix_long.groupby("global_id").size().reset_index(name="boltz_helical_wheel_count_long")

# # Merge all
# helix_agg = helix_agg_all.merge(helix_len_stats, on="global_id", how="left")
# helix_agg = helix_agg.merge(helix_counts, on="global_id", how="left")
# helix_agg = helix_agg.merge(helix_agg_long, on="global_id", how="left")
# helix_agg = helix_agg.merge(helix_counts_long, on="global_id", how="left")

# # Fill NaN for binders with no helices (overall stats)
# helix_agg["boltz_max_hydrophobic_moment"] = helix_agg["boltz_max_hydrophobic_moment"].fillna(0)
# helix_agg["boltz_mean_hydrophobic_moment"] = helix_agg["boltz_mean_hydrophobic_moment"].fillna(0)
# helix_agg["boltz_min_hydrophobic_moment"] = helix_agg["boltz_min_hydrophobic_moment"].fillna(0)
# helix_agg["boltz_max_charge_moment"] = helix_agg["boltz_max_charge_moment"].fillna(0)
# helix_agg["boltz_mean_charge_moment"] = helix_agg["boltz_mean_charge_moment"].fillna(0)
# helix_agg["boltz_min_charge_moment"] = helix_agg["boltz_min_charge_moment"].fillna(0)
# helix_agg["boltz_max_net_charge_total"] = helix_agg["boltz_max_net_charge_total"].fillna(0)
# helix_agg["boltz_mean_net_charge_total"] = helix_agg["boltz_mean_net_charge_total"].fillna(0)
# helix_agg["boltz_min_net_charge_total"] = helix_agg["boltz_min_net_charge_total"].fillna(0)
# helix_agg["boltz_max_helix_length"] = helix_agg["boltz_max_helix_length"].fillna(0)
# helix_agg["boltz_mean_helix_length"] = helix_agg["boltz_mean_helix_length"].fillna(0)
# helix_agg["boltz_total_helix_length"] = helix_agg["boltz_total_helix_length"].fillna(0)
# helix_agg["boltz_helical_wheel_count"] = helix_agg["boltz_helical_wheel_count"].fillna(0).astype(int)

# # Fill NaN for binders with no long helices (long stats)
# helix_agg["boltz_max_hydrophobic_moment_long"] = helix_agg["boltz_max_hydrophobic_moment_long"].fillna(0)
# helix_agg["boltz_mean_hydrophobic_moment_long"] = helix_agg["boltz_mean_hydrophobic_moment_long"].fillna(0)
# helix_agg["boltz_min_hydrophobic_moment_long"] = helix_agg["boltz_min_hydrophobic_moment_long"].fillna(0)
# helix_agg["boltz_max_charge_moment_long"] = helix_agg["boltz_max_charge_moment_long"].fillna(0)
# helix_agg["boltz_mean_charge_moment_long"] = helix_agg["boltz_mean_charge_moment_long"].fillna(0)
# helix_agg["boltz_min_charge_moment_long"] = helix_agg["boltz_min_charge_moment_long"].fillna(0)
# helix_agg["boltz_max_net_charge_total_long"] = helix_agg["boltz_max_net_charge_total_long"].fillna(0)
# helix_agg["boltz_mean_net_charge_total_long"] = helix_agg["boltz_mean_net_charge_total_long"].fillna(0)
# helix_agg["boltz_min_net_charge_total_long"] = helix_agg["boltz_min_net_charge_total_long"].fillna(0)
# helix_agg["boltz_helical_wheel_count_long"] = helix_agg["boltz_helical_wheel_count_long"].fillna(0).astype(int)


# # remove global_id 11985 and shift down by 1 for all higher ids to account for missing binder11985
# helix_agg = helix_agg[helix_agg["global_id"] != 11985]
# helix_agg.loc[helix_agg["global_id"] > 11985, "global_id"] -= 1

# metrics_df = metrics_df.merge(helix_agg, on="global_id", how="left")

# Experimental Results

In [31]:
import pandas as pd

results_df = pd.read_csv("../data/good/master_data_12k.csv", usecols=["global_id", "sequence"])

In [32]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_comb_raw_read_counts.csv")
df["global_id"] = df["global_id"].astype(int)

# prepend with leah_12k_ to all columns except global_id
df = df.rename(columns={col: f"leah_12k_{col}" if col != "global_id" else col for col in df.columns})
results_df = results_df.merge(df, on="global_id", how="left")

In [33]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_dna.csv")
df["global_id"] = df["global_id"].astype(int)
df = df[["global_id", "full_optimized_sequence"]]
df = df.rename(columns={"full_optimized_sequence": "dna_sequence"})
results_df = results_df.merge(df, on="global_id", how="left")

In [34]:
# NOTE: Good!
df = pd.read_csv("../data/good/Q-424341_leah_normalized_count_v2.csv")

results_df = results_df.merge(df, on="dna_sequence", how="left")

In [35]:
# NOTE: Good!
df = pd.read_csv("../data/good/leah_12k_edgeR_statistics.csv")
df = df.rename(columns={"SequenceID": "global_id"})
df["global_id"] = df["global_id"].astype(int)

# prepend with leah_12k_ to all columns except global_id
df = df.rename(columns={col: f"leah_12k_{col}" if col != "global_id" else col for col in df.columns})
df = df.rename(columns={"leah_12k_2-fold threshold?": "leah_12k_2fold_threshold", "leah_12k_Significant?": "leah_12k_significant"})

results_df = results_df.merge(df, on="global_id", how="left")

results_df["leah_12k_detected"] = ~results_df["leah_12k_2fold_threshold"].isna()

In [36]:
# NOTE: Good! Only the top 10 teams
df = pd.read_csv("../data/good/master_data_top10.csv")
df = df.iloc[2:].reset_index(drop=True) # remove + and - controls
df = df.drop(columns=["team", "Final Ranking"])

# global_id	Cytotoxicity AUC (R: CD20)	Cytotoxicity AUC (K: Control)	Fold Change in CTV MFI AUC (Proliferation)	Cytotoxicity AUC (R-K)	% Cytokine+	Fold Change in Total Cell Count/Well (Expansion)	Cytotoxicity AUC (R-K) MinMax Norm	% Cytokine+ MinMax Norm	Fold Change in Total Cell Count/Well (Expansion) MinMax Norm	Sum of Norms

df.columns = ["global_id", "cytotox_AUC_R_CD20", "cytotox_AUC_K_Control", "fold_change_CTV_MFI_AUC_proliferation", "cytotox_AUC_R_minus_K", "percent_cytokine_positive", "fold_change_total_cell_count_per_well_expansion", "cytotox_AUC_R_minus_K_minmax_norm", "percent_cytokine_positive_minmax_norm", "fold_change_total_cell_count_per_well_expansion_minmax_norm", "sum_of_norms"]
# add leah_top10_ prefix to all columns except global_id
df = df.rename(columns={col: f"leah_top10_{col}" if col != "global_id" else col for col in df.columns})
df["global_id"] = df["global_id"].astype(int)

results_df = results_df.merge(df, on="global_id", how="left")

In [37]:
# computed metrics
def map_four_class_int_to_label(v):
    if pd.isna(v):
        return "Not Recovered"
    if v == 0:
        return "Not Sig"
    if v == 1:
        return "Up"
    if v == -1:
        return "Down"
    return "Not Recovered"

results_df["leah_12k_isup"] = results_df["leah_12k_2fold_threshold"] == "Up"
results_df["leah_12k_twist_dna_detected"] = results_df["twist_dna_read_percentile"].apply(lambda x: x > 0)
results_df["leah_12k_2fold_threshold_int"] = results_df["leah_12k_2fold_threshold"].map(
    {"Up": 1, "Not Sig": 0, "Down": -1, np.nan: np.nan}
)
results_df["fourclass"] = results_df["leah_12k_2fold_threshold_int"].apply(map_four_class_int_to_label)


# More computed metrics

In [38]:
# compute shit ton of metrics here instead of elsewhere
metrics_df = metrics_df.sort_values("global_id").reset_index(drop=True)
results_df = results_df.sort_values("global_id").reset_index(drop=True)

def compute_human_frequencies():
    human_freqs = Counter()
    total_aa = 0
    with open("../data/homo_sapiens_UP000005640_9606.fasta", "r") as f:
        seq_parts = []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq_parts:
                    seq = "".join(seq_parts)
                    human_freqs.update(seq)
                    total_aa += len(seq)
                    seq_parts = []
            else:
                seq_parts.append(line)

        if seq_parts:
            seq = "".join(seq_parts)
            human_freqs.update(seq)
            total_aa += len(seq)

    freqs = {aa: human_freqs[aa] / total_aa for aa in human_freqs}
    return freqs

human_freqs = compute_human_frequencies()    

def kl_between_seq_and_human(seq):
    """
    Compute KL divergence between the amino acid composition of seq
    and the human proteome frequencies.
    """
    pa = ProteinAnalysis(seq)
    freqs = pa.get_amino_acids_percent()

    kl_div = 0.0
    for aa in human_freqs.keys():
        p = freqs.get(aa, 0.0)
        q = human_freqs[aa]
        if p > 0:
            kl_div += p * math.log2(p / q)
    return kl_div

def compute_gravy(seq: str) -> float:
    """
    Compute GRAVY (Grand Average of Hydropathy) for an amino acid sequence.
    """
    analyzed_seq = ProteinAnalysis(seq)
    return analyzed_seq.gravy()

def sequence_complexity(seq):
    """
    Shannon entropy over amino acid frequencies.
    Higher entropy means higher complexity.
    """
    pa = ProteinAnalysis(seq)
    freqs = pa.get_amino_acids_percent()

    entropy = 0.0
    for aa, p in freqs.items():
        if p > 0:
            entropy += -p * math.log2(p)
    return entropy

def dna_sequence_entropy(seq):
    """
    Shannon entropy over nucleotide frequencies.
    Higher entropy means higher complexity.
    """
    seq = seq.upper()
    length = len(seq)
    freqs = Counter(seq)
    
    entropy = 0.0
    for nucleotide, count in freqs.items():
        p = count / length
        if p > 0:
            entropy += -p * math.log2(p)
    return entropy

def num_cysteines(seq):
    """
    Count the number of cysteine residues in the sequence.
    """
    return seq.count('C')

def compute_charge(seq: str) -> float:
    """
    Compute the net charge of an amino acid sequence at pH 7.0.
    """
    analyzed_seq = ProteinAnalysis(seq)
    return analyzed_seq.charge_at_pH(7.0)

def max_charge_of_len_twenty(seq: str) -> float:
    """
    Compute the maximum net charge of any contiguous subsequence of length 20
    in the amino acid sequence at pH 7.0.
    """
    max_charge = float('-inf')
    for i in range(len(seq) - 19):
        subseq = seq[i:i + 20]
        analyzed_subseq = ProteinAnalysis(subseq)
        charge = analyzed_subseq.charge_at_pH(7.0)
        if charge > max_charge:
            max_charge = charge
    return max_charge if max_charge != float('-inf') else 0.0

def min_charge_of_len_twenty(seq: str) -> float:
    """
    Compute the minimum net charge of any contiguous subsequence of length 20
    in the amino acid sequence at pH 7.0.
    """
    min_charge = float('inf')
    for i in range(len(seq) - 19):
        subseq = seq[i:i + 20]
        analyzed_subseq = ProteinAnalysis(subseq)
        charge = analyzed_subseq.charge_at_pH(7.0)
        if charge < min_charge:
            min_charge = charge
    return min_charge if min_charge != float('inf') else 0.0

def max_gravy_of_len_twenty(seq: str) -> float:
    """
    Compute the maximum GRAVY of any contiguous subsequence of length 20
    in the amino acid sequence.
    """
    max_gravy = float('-inf')
    for i in range(len(seq) - 19):
        subseq = seq[i:i + 20]
        analyzed_subseq = ProteinAnalysis(subseq)
        gravy = analyzed_subseq.gravy()
        if gravy > max_gravy:
            max_gravy = gravy
    return max_gravy if max_gravy != float('-inf') else 0.0

def min_gravy_of_len_twenty(seq: str) -> float:
    """
    Compute the minimum GRAVY of any contiguous subsequence of length 20
    in the amino acid sequence.
    """
    min_gravy = float('inf')
    for i in range(len(seq) - 19):
        subseq = seq[i:i + 20]
        analyzed_subseq = ProteinAnalysis(subseq)
        gravy = analyzed_subseq.gravy()
        if gravy < min_gravy:
            min_gravy = gravy
    return min_gravy if min_gravy != float('inf') else 0.0

def gc_content(seq: str) -> float:
    """
    Compute the GC content of a DNA sequence.
    """
    g_count = seq.count('G')
    c_count = seq.count('C')
    gc_count = g_count + c_count
    return gc_count / len(seq) if len(seq) > 0 else 0.0

def contains_linker(seq):
    linker_seqs = ["GGGGS", "GGGS", "GGS", "GGGGGS"]
    for linker in linker_seqs:
        if linker in seq:
            return True
    return False

def longest_duplicated_substring(seq):
    """
    Find the longest duplicated substring in the sequence such that
    the two defining occurrences do not overlap.

    Return: (length, substring, total_occurrences)
    where total_occurrences counts all (possibly overlapping) matches.
    """
    def count_overlapping(haystack, needle):
        if not needle:
            return 0
        count = 0
        i = 0
        while True:
            i = haystack.find(needle, i)
            if i == -1:
                break
            count += 1
            i += 1  # move by 1 to allow overlaps
        return count

    n = len(seq)
    suffixes = sorted((seq[i:], i) for i in range(n))

    max_len = 0
    best_substring = ""

    for i in range(1, n):
        s1, idx1 = suffixes[i - 1]
        s2, idx2 = suffixes[i]

        # compute LCP between neighboring suffixes
        j = 0
        limit = min(len(s1), len(s2))
        while j < limit and s1[j] == s2[j]:
            j += 1

        # prevent overlap between the two suffixes used to define the match
        distance = abs(idx1 - idx2)
        lcp_no_overlap = min(j, distance)

        if lcp_no_overlap > max_len:
            max_len = lcp_no_overlap
            start = min(idx1, idx2)
            best_substring = seq[start:start + max_len]

    if max_len == 0:
        return 0, "", 0

    total_occurrences = count_overlapping(seq, best_substring)
    return max_len, best_substring, total_occurrences

def longest_stretch_of_(seq, char):
    max_len = 0
    current_len = 0
    for c in seq:
        if c == char:
            current_len += 1
            if current_len > max_len:
                max_len = current_len
        else:
            current_len = 0
    return max_len

def ratio_of_letter(seq, letters):
    seq = seq.upper()
    length = len(seq)
    count = sum(seq.count(letter) for letter in letters)
    return count / length if length > 0 else 0.0

alpha_codes = {"H", "G", "I"}
beta_codes = {"E", "B"}
kae_set = {"K", "A", "E"}
ke_set = {"K", "E"}

def compute_ss_metrics(row):
    seq = row["sequence"]
    dssp = row["dssp"]

    if not isinstance(seq, str) or not isinstance(dssp, str):
        return pd.Series(
            {
                "dssp_alpha_ratio": np.nan,
                "dssp_beta_ratio": np.nan,
                "dssp_other_ratio": np.nan,
                "dssp_ke_alpha_ratio": np.nan,
                "dssp_kae_alpha_ratio": np.nan,
            }
        )

    L = min(len(seq), len(dssp))
    if L == 0:
        return pd.Series(
            {
                "dssp_alpha_ratio": np.nan,
                "dssp_beta_ratio": np.nan,
                "dssp_other_ratio": np.nan,
                "dssp_ke_alpha_ratio": np.nan,
                "dssp_kae_alpha_ratio": np.nan,
            }
        )

    alpha = 0
    beta = 0
    other = 0
    kae_alpha = 0
    ke_alpha = 0

    for aa, ss in zip(seq[:L], dssp[:L]):
        if ss in alpha_codes:
            alpha += 1
            if aa in kae_set:
                kae_alpha += 1
            if aa in ke_set:
                ke_alpha += 1
        elif ss in beta_codes:
            beta += 1
        else:
            other += 1

    total = alpha + beta + other

    alpha_ratio = alpha / total if total > 0 else np.nan
    beta_ratio = beta / total if total > 0 else np.nan
    other_ratio = other / total if total > 0 else np.nan

    kae_alpha_ratio = (
        kae_alpha / total if total > 0 else np.nan
    )

    ke_alpha_ratio = (
        ke_alpha / total if total > 0 else np.nan
    )

    return pd.Series(
        {
            "dssp_alpha_ratio": alpha_ratio,
            "dssp_beta_ratio": beta_ratio,
            "dssp_other_ratio": other_ratio,
            "dssp_ke_alpha_ratio": ke_alpha_ratio,
            "dssp_kae_alpha_ratio": kae_alpha_ratio,
        }
    )

def count_poly_A_stretches(dna_seq):
    """Count stretches of 3+ consecutive A's in DNA sequence"""
    if pd.isna(dna_seq):
        return np.nan
    count = 0
    current_stretch = 0
    for base in str(dna_seq):
        if base == 'A':
            current_stretch += 1
            if current_stretch >= 2:
                count += 1
        else:
            current_stretch = 0
    return count

def count_poly_E_stretches(aa_seq):
    """Count stretches of 3+ consecutive E's in AA sequence"""
    if pd.isna(aa_seq):
        return np.nan
    count = 0
    current_stretch = 0
    for aa in str(aa_seq):
        if aa == 'E':
            current_stretch += 1
            if current_stretch >= 2:
                count += 1
        else:
            current_stretch = 0
    return count

metrics_to_max = [
 'boltz_A_B_ipSAE',
 'boltz_A_B_ipSAE_d0chn',
 'boltz_A_B_ipSAE_d0chn_max',
 'boltz_A_B_ipSAE_d0dom',
 'boltz_A_B_ipSAE_d0dom_max',
 'boltz_A_B_ipSAE_max',
 'boltz_A_B_lis_score',
 'boltz_A_B_lis_score_max',
 'boltz_A_B_pdockq',
 'boltz_A_B_pdockq2',
 'boltz_A_B_pdockq2_max',
 'boltz_A_B_pdockq_max',
 'boltz_A_C_ipSAE',
 'boltz_A_C_ipSAE_d0chn',
 'boltz_A_C_ipSAE_d0chn_max',
 'boltz_A_C_ipSAE_d0dom',
 'boltz_A_C_ipSAE_d0dom_max',
 'boltz_A_C_ipSAE_max',
 'boltz_A_C_lis_score',
 'boltz_A_C_lis_score_max',
 'boltz_A_C_pdockq',
 'boltz_A_C_pdockq2',
 'boltz_A_C_pdockq2_max',
 'boltz_A_C_pdockq_max',
 'chai1_A_B_ipSAE',
 'chai1_A_B_ipSAE_d0chn',
 'chai1_A_B_ipSAE_d0chn_max',
 'chai1_A_B_ipSAE_max',
 'chai1_A_B_lis_score',
 'chai1_A_B_lis_score_max',
 'chai1_A_B_pdockq',
 'chai1_A_B_pdockq2',
 'chai1_A_B_pdockq2_max',
 'chai1_A_B_pdockq_max',
 'chai1_A_C_ipSAE',
 'chai1_A_C_ipSAE_d0chn',
 'chai1_A_C_ipSAE_d0chn_max',
 'chai1_A_C_ipSAE_max',
 'chai1_A_C_lis_score',
 'chai1_A_C_lis_score_max',
 'chai1_A_C_pdockq',
 'chai1_A_C_pdockq2',
 'chai1_A_C_pdockq2_max',
 'chai1_A_C_pdockq_max',
 'esmfold_start_A_B_ipSAE',
 'esmfold_start_A_B_ipSAE_d0chn',
 'esmfold_start_A_B_ipSAE_d0chn_max',
 'esmfold_start_A_B_ipSAE_max',
 'esmfold_start_A_B_lis_score',
 'esmfold_start_A_B_lis_score_max',
 'esmfold_start_A_B_pdockq',
 'esmfold_start_A_B_pdockq2',
 'esmfold_start_A_B_pdockq2_max',
 'esmfold_start_A_B_pdockq_max',
 'esmfold_start_A_C_ipSAE',
 'esmfold_start_A_C_ipSAE_d0chn',
 'esmfold_start_A_C_ipSAE_d0chn_max',
 'esmfold_start_A_C_ipSAE_max',
 'esmfold_start_A_C_lis_score',
 'esmfold_start_A_C_lis_score_max',
 'esmfold_start_A_C_pdockq',
 'esmfold_start_A_C_pdockq2',
 'esmfold_start_A_C_pdockq2_max',
 'esmfold_start_A_C_pdockq_max',
 'esmfold_start_glycine_linker_A_B_ipSAE',
 'esmfold_start_glycine_linker_A_B_ipSAE_d0chn',
 'esmfold_start_glycine_linker_A_B_ipSAE_d0chn_max',
 'esmfold_start_glycine_linker_A_B_ipSAE_max',
 'esmfold_start_glycine_linker_A_B_lis_score',
 'esmfold_start_glycine_linker_A_B_lis_score_max',
 'esmfold_start_glycine_linker_A_B_pdockq',
 'esmfold_start_glycine_linker_A_B_pdockq2',
 'esmfold_start_glycine_linker_A_B_pdockq2_max',
 'esmfold_start_glycine_linker_A_B_pdockq_max',
 'esmfold_start_glycine_linker_A_C_ipSAE',
 'esmfold_start_glycine_linker_A_C_ipSAE_d0chn',
 'esmfold_start_glycine_linker_A_C_ipSAE_d0chn_max',
 'esmfold_start_glycine_linker_A_C_ipSAE_max',
 'esmfold_start_glycine_linker_A_C_lis_score',
 'esmfold_start_glycine_linker_A_C_lis_score_max',
 'esmfold_start_glycine_linker_A_C_pdockq',
 'esmfold_start_glycine_linker_A_C_pdockq2',
 'esmfold_start_glycine_linker_A_C_pdockq2_max',
 'esmfold_start_glycine_linker_A_C_pdockq_max',
]

for col in metrics_to_max:
    if "_A_B_" in col:
        col_c = col.replace("_A_B_", "_A_C_")
        new_col = col.replace("_A_B_", "_A_BC_max_")
        
        if col in metrics_df.columns and col_c in metrics_df.columns:
             metrics_df[new_col] = metrics_df[[col, col_c]].max(axis=1)

metrics_df["dna_sequence"] = results_df["dna_sequence"]
metrics_df["gravy_score"] = metrics_df["sequence"].apply(compute_gravy)
metrics_df["aa_sequence_entropy"] = metrics_df["sequence"].apply(sequence_complexity)
metrics_df["dna_sequence_entropy"] = metrics_df["dna_sequence"].apply(dna_sequence_entropy)
metrics_df["num_cysteines"] = metrics_df["sequence"].apply(num_cysteines)
metrics_df["seq_kl_vs_human"] = metrics_df["sequence"].apply(kl_between_seq_and_human)
metrics_df["seq_charge"] = metrics_df["sequence"].apply(compute_charge)
metrics_df["max_charge_len_20"] = metrics_df["sequence"].apply(max_charge_of_len_twenty)
metrics_df["min_charge_len_20"] = metrics_df["sequence"].apply(min_charge_of_len_twenty)
metrics_df["longest_stretch_of_K"] = metrics_df["sequence"].apply(lambda seq: longest_stretch_of_(seq, 'K'))
metrics_df["longest_stretch_of_E"] = metrics_df["sequence"].apply(lambda seq: longest_stretch_of_(seq, 'E'))
metrics_df["longest_stretch_of_A"] = metrics_df["sequence"].apply(lambda seq: longest_stretch_of_(seq, 'A'))
metrics_df["max_gravy_len_20"] = metrics_df["sequence"].apply(max_gravy_of_len_twenty)
metrics_df["min_gravy_len_20"] = metrics_df["sequence"].apply(min_gravy_of_len_twenty)
metrics_df["dna_gc_content"] = metrics_df["dna_sequence"].apply(gc_content)
metrics_df["is_linker"] = metrics_df["sequence"].apply(contains_linker)
metrics_df[["longest_dup_substr_len", "longest_dup_substr", "longest_dup_occurrences"]] = metrics_df["sequence"].apply(
    lambda seq: pd.Series(longest_duplicated_substring(seq))
)
metrics_df["total_duplicated_residues"] = metrics_df["longest_dup_substr_len"] * metrics_df["longest_dup_occurrences"]
# repeat for dna
metrics_df[["longest_dup_substr_len_dna", "longest_dup_substr_dna", "longest_dup_occurrences_dna"]] = metrics_df["dna_sequence"].apply(
    lambda seq: pd.Series(longest_duplicated_substring(seq))
)
metrics_df["total_duplicated_nucleotides"] = metrics_df["longest_dup_substr_len_dna"] * metrics_df["longest_dup_occurrences_dna"]
metrics_df["ratio_K"] = metrics_df["sequence"].apply(lambda seq: ratio_of_letter(seq, ['K']))
metrics_df["ratio_E"] = metrics_df["sequence"].apply(lambda seq: ratio_of_letter(seq, ['E']))
metrics_df["ratio_KE"] = metrics_df["sequence"].apply(lambda seq: ratio_of_letter(seq, ['K', 'E']))
metrics_df[["dssp_alpha_ratio",
            "dssp_beta_ratio",
            "dssp_other_ratio",
            "dssp_ke_alpha_ratio",
            "dssp_kae_alpha_ratio"]] = metrics_df.apply(
    compute_ss_metrics, axis=1
)
metrics_df["num_A_repeats_dna"] = metrics_df["dna_sequence"].apply(count_poly_A_stretches)
metrics_df["num_E_repeats_aa"] = metrics_df["sequence"].apply(count_poly_E_stretches)

results_df = pd.merge(results_df, metrics_df[["global_id", "team"]], on="global_id", how="left")

/stor/scratch/Ellington/archive/cwk687/home/micromamba/envs/dev/lib/python3.11/site-packages/Bio/SeqUtils/ProtParam.py:106: BiopythonDeprecationWarning: The get_amino_acids_percent method has been deprecated and will likely be removed from Biopython in the near future. Please use the amino_acids_percent attribute instead.
  warnings.warn(
/stor/scratch/Ellington/archive/cwk687/home/micromamba/envs/dev/lib/python3.11/site-packages/Bio/SeqUtils/ProtParam.py:106: BiopythonDeprecationWarning: The get_amino_acids_percent method has been deprecated and will likely be removed from Biopython in the near future. Please use the amino_acids_percent attribute instead.
  warnings.warn(


In [39]:
metrics_df.to_csv("../data/12k_all_metrics.csv", index=False)
results_df.to_csv("../data/12k_all_results.csv", index=False)